# Section 6 — DSCMI Decoder: Depthwise Separable Convolution with Multi-Head Sparse Self-Attention

**Paper:** Fu et al. 2025, *"VR Paradigm-Agnostic Motor Imagery Decoding Using Lightweight Network With Adaptive Attention Mechanism"*, Journal of Medical Systems 49:152.

This notebook implements:
1. **Full preprocessing pipeline** (bandpass 7–35 Hz, ICA artifact removal, epoching)
2. **DSCMI decoder** — depthwise separable convolution + multi-head sparse self-attention (Eqs. 1–10, Algorithm 1)
3. **Automated subject-specific hyperparameter selection** (Bayesian optimization over head count & sparse coefficient c)
4. **Real-time EEG simulation module** — hardware-agnostic closed-loop BCI replay

**Dataset:** PhysioNet EEG Motor Movement/Imagery (Schalk et al.), runs 4/8/12, T1=left fist MI, T2=right fist MI.

## Section 6.0 — Environment Setup & Imports

In [ ]:
# ── Install required packages (Colab-compatible) ──
import subprocess, sys

def install(pkg):
    """Install a package if not already available."""
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['mne', 'moabb', 'torch', 'scikit-learn', 'matplotlib', 'seaborn',
            'scipy', 'pandas', 'optuna']:  # optuna for Bayesian hyperparameter search
    try:
        __import__(pkg.replace('-', '_').split('==')[0])
    except ImportError:
        install(pkg)

print("All packages available.")

In [ ]:
# ── Standard library ──
import os, time, copy, warnings, math, threading, queue
from pathlib import Path
from collections import defaultdict

# ── Numerical / scientific ──
import numpy as np                          # array operations for EEG matrices
import scipy.signal as sig                   # digital filtering utilities
from scipy.stats import ttest_rel            # paired t-test for significance testing

# ── Visualization ──
import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50  # suppress figure warnings
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')                   # clean publication-style plots

# ── Data handling ──
import pandas as pd

# ── EEG processing ──
import mne                                   # MNE-Python: EEG I/O, filtering, ICA, epoching
mne.set_log_level('WARNING')                 # reduce verbose output

# ── Machine learning ──
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import (accuracy_score, cohen_kappa_score,
                             confusion_matrix, classification_report,
                             roc_curve, auc)
from sklearn.preprocessing import label_binarize

# ── Deep learning ──
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

# ── Hyperparameter optimization ──
import optuna                                # Bayesian TPE sampler for automated tuning
optuna.logging.set_verbosity(optuna.logging.WARNING)  # quiet output

# ── Reproducibility seeds ──
SEED = 42
np.random.seed(SEED)                         # numpy RNG seed
torch.manual_seed(SEED)                      # pytorch CPU seed
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)          # pytorch GPU seed
    torch.backends.cudnn.deterministic = True  # deterministic cuDNN ops

# ── Device selection ──
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# ── Suppress non-critical warnings ──
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ── Output directory for figures ──
FIG_DIR = Path('figures'); FIG_DIR.mkdir(exist_ok=True)

print("Section 6.0 complete — environment ready.")

## Section 6.1 — Data Loading (PhysioNet Motor Imagery)

We load runs 4, 8, 12 (motor imagery: T1 = left fist, T2 = right fist) from the PhysioNet EEG Motor Movement/Imagery dataset. The paper uses this dataset with 64 channels sampled at 160 Hz.

In [ ]:
def load_physionet_subject(subject_id, runs=(4, 8, 12)):
    """
    Load raw EEG data for one PhysioNet subject.

    WHY:  The PhysioNet MI dataset (Schalk et al., Goldberger et al. 2000) contains
          64-channel EEG at 160 Hz from 109 subjects performing motor imagery.
    WHAT: Downloads (if needed) and concatenates specified runs into a single Raw object.
    INPUT:
        subject_id : int — subject number (1–109)
        runs : tuple of int — run numbers (4,8,12 = MI left/right fist)
    OUTPUT:
        raw : mne.io.Raw — concatenated raw EEG
        events : ndarray shape (n_events, 3) — MNE event array
        event_id : dict — mapping of event names to codes
    """
    # mne.datasets.eegbci downloads PhysioNet EDF files into ~/mne_data/
    fnames = mne.datasets.eegbci.load_data(subject_id, runs, verbose=False)
    
    # Read each EDF and concatenate into one continuous Raw object
    raws = [mne.io.read_raw_edf(f, preload=True, verbose=False) for f in fnames]
    raw = mne.concatenate_raws(raws)  # merges along time axis
    
    # Standardize channel names to 10-20 system (PhysioNet uses "Fc5." style)
    mne.datasets.eegbci.standardize(raw)
    
    # Set the standard 10-05 montage for topographic plotting
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage, on_missing='ignore')  # ignore channels not in montage
    
    # Extract events from annotations (T0=rest, T1=left fist MI, T2=right fist MI)
    events, event_id_all = mne.events_from_annotations(raw, verbose=False)
    
    # We only need T1 and T2 (motor imagery classes)
    # Map annotation strings to integer codes — PhysioNet uses 'T1'=2, 'T2'=3
    event_id = {}
    for k, v in event_id_all.items():
        if 'T1' in k:  event_id['left_fist'] = v   # T1 = left fist MI
        elif 'T2' in k: event_id['right_fist'] = v  # T2 = right fist MI
    
    return raw, events, event_id

# Load first subject as demonstration
raw_demo, events_demo, event_id_demo = load_physionet_subject(1)
print(f"Channels: {len(raw_demo.ch_names)}, Sfreq: {raw_demo.info['sfreq']} Hz")
print(f"Events: {event_id_demo}, Total events: {len(events_demo)}")
print("Section 6.1 complete — data loading verified.")

## Section 6.2 — Preprocessing Pipeline

Following the paper:
1. **Bandpass filter** 7–35 Hz (FIR) — removes baseline drift (<7 Hz) and high-freq muscle artifacts (>35 Hz)
2. **ICA** artifact removal — identify & remove eye-blink and muscle components
3. **Epoching** — segment continuous EEG into trials aligned to event onsets
4. **Normalization** — z-score per channel per trial for network input stability

In [ ]:
def preprocess_raw(raw, events, event_id, tmin=0.0, tmax=4.0,
                   l_freq=7.0, h_freq=35.0, apply_ica=True):
    """
    Full preprocessing pipeline as described in Fu et al. 2025.

    WHY:  Raw EEG contains eye-blink artifacts, muscle noise, and baseline drift.
          Bandpass + ICA removes these while preserving mu (8-12 Hz) and beta (13-30 Hz)
          rhythms critical for motor imagery decoding.
    WHAT: Filter → ICA → Epoch → Extract numpy arrays.
    INPUT:
        raw : mne.io.Raw — continuous EEG data
        events : ndarray (n_events, 3) — event timing array
        event_id : dict — event name → code mapping
        tmin, tmax : float — epoch window relative to event onset (seconds)
        l_freq, h_freq : float — bandpass cutoff frequencies (Hz)
        apply_ica : bool — whether to run ICA artifact removal
    OUTPUT:
        X : ndarray (n_trials, n_channels, n_timepoints) — preprocessed EEG epochs
        y : ndarray (n_trials,) — integer labels (0 = left fist, 1 = right fist)
        epochs : mne.Epochs — MNE epochs object for further analysis
    """
    raw_copy = raw.copy()  # avoid modifying the original
    
    # ── Step 1: Bandpass filter (FIR, 7-35 Hz) ──
    # Paper Sec. III: "finite impulse response bandpass filter with cutoff frequencies
    # of 7 Hz and 35 Hz... preserving relevant brain activity"
    raw_copy.filter(l_freq=l_freq, h_freq=h_freq,
                    method='fir',           # FIR as specified in paper
                    fir_design='firwin',     # window-function method (paper: "designed using a window function method")
                    verbose=False)
    
    # ── Step 2: ICA artifact removal ──
    # Paper: "ICA was applied to eliminate ocular and muscular artifacts"
    if apply_ica:
        # Set EEG channel type explicitly for ICA
        raw_copy.set_eeg_reference('average', projection=True, verbose=False)
        raw_copy.apply_proj(verbose=False)  # apply average reference
        
        # Fit ICA — use enough components to capture most variance
        n_components = min(20, len(raw_copy.ch_names) - 1)  # cap at 20 for speed
        ica = mne.preprocessing.ICA(
            n_components=n_components,
            method='fastica',              # FastICA for speed
            random_state=SEED,
            max_iter=500,                  # sufficient convergence iterations
            verbose=False
        )
        ica.fit(raw_copy, verbose=False)   # decompose EEG into independent components
        
        # Auto-detect EOG-like components using frontal channels (Fp1, Fp2, Fpz)
        # These channels pick up eye blinks with high amplitude
        eog_channels = [ch for ch in raw_copy.ch_names if ch.lower().startswith('fp')]
        if not eog_channels:
            eog_channels = [raw_copy.ch_names[0]]  # fallback: use first channel
        
        # Find components correlated with EOG channels
        eog_indices = []
        for eog_ch in eog_channels:
            try:
                indices, scores = ica.find_bads_eog(
                    raw_copy, ch_name=eog_ch, verbose=False
                )
                eog_indices.extend(indices)
            except Exception:
                pass  # channel might not work for correlation
        
        # Remove duplicates and apply — subtract artifact components from signal
        ica.exclude = list(set(eog_indices))
        raw_copy = ica.apply(raw_copy, verbose=False)
    else:
        # At minimum, apply average reference for consistent spatial filtering
        raw_copy.set_eeg_reference('average', projection=True, verbose=False)
        raw_copy.apply_proj(verbose=False)
    
    # ── Step 3: Epoching ──
    # Segment continuous EEG into trial-locked windows
    # PhysioNet: each trial lasts 4s, we take [tmin, tmax] relative to onset
    epochs = mne.Epochs(
        raw_copy, events, event_id,
        tmin=tmin, tmax=tmax,
        baseline=None,                     # no baseline correction (paper doesn't specify)
        preload=True,
        verbose=False
    )
    epochs.drop_bad(verbose=False)          # remove epochs with extreme amplitudes
    
    # ── Step 4: Extract arrays and normalize ──
    X = epochs.get_data(copy=True)           # shape: (n_trials, n_channels, n_timepoints)
    
    # Map event labels to 0-indexed integers for classification
    labels = epochs.events[:, -1]            # event codes from MNE
    unique_labels = sorted(set(labels))
    label_map = {old: new for new, old in enumerate(unique_labels)}
    y = np.array([label_map[l] for l in labels])  # 0 = left fist, 1 = right fist
    
    # Z-score normalization per channel per trial
    # WHY: neural networks converge faster with zero-mean, unit-variance inputs
    for i in range(X.shape[0]):
        for ch in range(X.shape[1]):
            mu = X[i, ch].mean()             # trial/channel mean
            sigma = X[i, ch].std() + 1e-8    # avoid division by zero
            X[i, ch] = (X[i, ch] - mu) / sigma  # standardize
    
    return X, y, epochs


# ── Preprocess demo subject ──
X_demo, y_demo, epochs_demo = preprocess_raw(raw_demo, events_demo, event_id_demo)
print(f"X shape: {X_demo.shape} (trials, channels, timepoints)")
print(f"y shape: {y_demo.shape}, classes: {np.unique(y_demo)}, counts: {np.bincount(y_demo)}")
print("Section 6.2 complete — preprocessing pipeline verified.")

In [ ]:
# ── Visualization: PSD before vs after filtering ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Before filtering — use original raw data
raw_unfilt = raw_demo.copy()  # unfiltered copy
psd_before = raw_unfilt.compute_psd(fmin=1, fmax=80, verbose=False)
psd_before.plot(axes=axes[0], show=False, spatial_colors=True)
axes[0].set_title('PSD — Before Filtering')

# After filtering — bandpass 7-35 Hz
raw_filt = raw_demo.copy().filter(7, 35, method='fir', fir_design='firwin', verbose=False)
psd_after = raw_filt.compute_psd(fmin=1, fmax=80, verbose=False)
psd_after.plot(axes=axes[1], show=False, spatial_colors=True)
axes[1].set_title('PSD — After 7–35 Hz Bandpass')

plt.tight_layout()
plt.savefig(FIG_DIR / 'section6_fig1_psd.png', dpi=150)
plt.show()

### What to look for in this plot:
- **Left (before):** Broad spectral content including low-frequency drift (<7 Hz) and 50/60 Hz line noise
- **Right (after):** Clean pass-band retaining only mu (8–12 Hz) and beta (13–30 Hz) rhythms critical for MI
- The filter should sharply attenuate frequencies outside 7–35 Hz

In [ ]:
# ── Visualization: Epoch averages per class at C3 and C4 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# C3 and C4 are key motor cortex channels — contralateral activation in MI
for idx, ch_name in enumerate(['C3', 'C4']):
    ch_idx = epochs_demo.ch_names.index(ch_name) if ch_name in epochs_demo.ch_names else 0
    times = epochs_demo.times  # time axis in seconds
    
    for cls, label_name in enumerate(['Left Fist', 'Right Fist']):
        mask = y_demo == cls  # boolean mask for this class
        mean_signal = X_demo[mask, ch_idx, :].mean(axis=0)  # trial-average
        axes[idx].plot(times, mean_signal, label=label_name, linewidth=1.5)
    
    axes[idx].set_title(f'Mean ERP at {ch_name}')
    axes[idx].set_xlabel('Time (s)')
    axes[idx].set_ylabel('Amplitude (z-scored)')
    axes[idx].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'section6_fig2_erp.png', dpi=150)
plt.show()

### What to look for in this plot:
- **C3 (left motor cortex):** Expect stronger ERD (amplitude suppression) for right-hand MI (contralateral)
- **C4 (right motor cortex):** Expect stronger ERD for left-hand MI (contralateral)
- Clear class separation at C3/C4 indicates good signal quality for decoding

## Section 6.3 — DSCMI Decoder Architecture

The DSCMI decoder (Fu et al. 2025, Fig. 3) consists of:
1. **Block1:** Temporal convolution — extracts frequency features along time axis
2. **Block2:** Depthwise convolution — learns spatial filters across channels, compresses channel dim to 1
3. **Block3:** Separable convolution — temporal-only convolution for higher-level feature mixing
4. **Squeeze:** Flatten & project to (192 × 512) feature matrix
5. **Multi-Head Sparse Self-Attention:** Redistribute feature weights using Algorithm 1
6. **Squeeze & Distill + Fully Connected:** Final classification head

In [ ]:
class SparseMultiHeadAttention(nn.Module):
    """
    Multi-Head Sparse Self-Attention (Algorithm 1, Eqs. 1-10 in Fu et al. 2025).

    WHY:  Standard self-attention has O(N^2) complexity. The sparse variant reduces this
          by selecting only the Top-u queries with highest skewness (Eq. 6), computing
          full attention only for those, and using mean(V) for the rest.
    WHAT: Implements QKV attention with sparse key sampling (Eq. 3) and query filtering
          (Eq. 5-6), then combines sparse attention output with mean values (Eq. 10).
    INPUT:
        x : Tensor (batch, seq_len, d_model) — feature matrix from conv backbone
    OUTPUT:
        out : Tensor (batch, seq_len, d_model) — attention-weighted features
    """
    def __init__(self, d_model, n_heads=8, c=0.5):
        """
        Args:
            d_model : int — feature dimension (must be divisible by n_heads)
            n_heads : int — number of attention heads
            c : float ∈ (0,1) — sparse parameter controlling how many queries to keep
        """
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model               # total feature dimension
        self.n_heads = n_heads               # number of parallel attention heads
        self.d_k = d_model // n_heads        # dimension per head
        self.c = c                           # sparse coefficient ∈ (0,1) — Eq. 5
        
        # Linear projections for Q, K, V — each projects d_model → d_model
        self.W_q = nn.Linear(d_model, d_model)  # query projection
        self.W_k = nn.Linear(d_model, d_model)  # key projection
        self.W_v = nn.Linear(d_model, d_model)  # value projection
        self.W_o = nn.Linear(d_model, d_model)  # output projection
    
    def forward(self, x):
        """
        Forward pass implementing Algorithm 1.
        
        INPUT: x — Tensor (batch, seq_len, d_model)
        OUTPUT: out — Tensor (batch, seq_len, d_model)
        """
        B, L, D = x.shape  # batch size, sequence length, feature dimension
        
        # ── Step 1: Project inputs to Q, K, V ──
        Q = self.W_q(x)  # (B, L, D)
        K = self.W_k(x)  # (B, L, D)
        V = self.W_v(x)  # (B, L, D)
        
        # ── Step 2: Reshape for multi-head attention ──
        # Split d_model into n_heads × d_k, then transpose for batch matmul
        Q = Q.view(B, L, self.n_heads, self.d_k).transpose(1, 2)  # (B, H, L, d_k)
        K = K.view(B, L, self.n_heads, self.d_k).transpose(1, 2)  # (B, H, L, d_k)
        V = V.view(B, L, self.n_heads, self.d_k).transpose(1, 2)  # (B, H, L, d_k)
        
        L_Q = L  # length of query
        L_K = L  # length of key
        
        # ── Step 3: Compute U — number of sampled keys (Eq. 3) ──
        # U = L_Q * ln(L_K) — controls how many keys we randomly sample
        U = max(1, int(L_Q * math.log(max(L_K, 2))))  # ensure at least 1
        U = min(U, L_K)  # can't sample more keys than available
        
        # Randomly sample U keys to form reduced key matrix K_bar (paper: Eq. 3-4)
        sample_idx = torch.randint(0, L_K, (U,), device=x.device)  # random key indices
        K_bar = K[:, :, sample_idx, :]  # (B, H, U, d_k) — sampled keys
        
        # ── Step 4: Compute attention score matrix S_bar (Eq. 4) ──
        # S_bar = Q @ K_bar^T — dot product between all queries and sampled keys
        S_bar = torch.matmul(Q, K_bar.transpose(-2, -1))  # (B, H, L_Q, U)
        
        # ── Step 5: Compute u — number of top queries to keep (Eq. 5) ──
        # u = c * ln(L_K) — sparse parameter controls query filtering aggressiveness
        u = max(1, int(self.c * math.log(max(L_K, 2))))  # at least 1 query
        u = min(u, L_Q)  # can't select more queries than available
        
        # ── Step 6: Compute skewness measure M for each query row (Eq. 6) ──
        # M_i = max(S_bar_i) - mean(S_bar_i) — measures how "peaked" the attention is
        # High M means this query has a strong preference → should be computed fully
        M = S_bar.max(dim=-1).values - S_bar.mean(dim=-1)  # (B, H, L_Q)
        
        # Select Top-u queries with highest skewness
        _, top_indices = M.topk(u, dim=-1)  # (B, H, u) — indices of most important queries
        
        # ── Step 7: Build filtered query matrix Q_tilde and recompute S_tilde (Eq. 7) ──
        # Gather the top-u query vectors
        top_indices_expanded = top_indices.unsqueeze(-1).expand(-1, -1, -1, self.d_k)  # (B,H,u,d_k)
        Q_tilde = torch.gather(Q, 2, top_indices_expanded)  # (B, H, u, d_k)
        
        # Recompute attention scores with filtered queries and ALL keys
        S_tilde = torch.matmul(Q_tilde, K.transpose(-2, -1))  # (B, H, u, L_K) — Eq. 7
        
        # ── Step 8: Compute V_u — sparse attention weighted values (Eq. 8) ──
        # V_u = softmax(Q_tilde @ K^T / sqrt(d_k)) @ V
        scale = math.sqrt(self.d_k)  # scaling factor to prevent dot products from growing large
        attn_weights = torch.softmax(S_tilde / scale, dim=-1)  # (B, H, u, L_K)
        V_u = torch.matmul(attn_weights, V)  # (B, H, u, d_k) — attended values for top queries
        
        # ── Step 9: Compute V_mean for remaining queries (Eq. 9) ──
        # Non-selected queries use mean of all values as their output
        V_mean = V.mean(dim=2, keepdim=True).expand(-1, -1, L, -1)  # (B, H, L, d_k)
        
        # ── Step 10: Combine V_u and V_mean (Eq. 10) ──
        # Va = V_mean + V_u (placed at their original positions)
        output = V_mean.clone()  # start with mean values everywhere
        
        # Scatter V_u values back to their original query positions
        top_indices_v = top_indices.unsqueeze(-1).expand(-1, -1, -1, self.d_k)  # (B,H,u,d_k)
        output.scatter_(2, top_indices_v, V_u + output.gather(2, top_indices_v))  # add V_u to V_mean at top positions
        
        # ── Reshape back and project ──
        output = output.transpose(1, 2).contiguous().view(B, L, self.d_model)  # (B, L, D)
        output = self.W_o(output)  # final linear projection
        
        return output


print("SparseMultiHeadAttention module defined.")

In [ ]:
class DSCMINet(nn.Module):
    """
    DSCMI Decoder — Depthwise Separable Convolution with Multi-Head Sparse Self-Attention.

    WHY:  Combines EEGNet-style depthwise separable convolutions (for efficient spatial-
          temporal feature extraction) with sparse self-attention (for adaptive feature
          weighting), yielding a lightweight yet powerful MI decoder.
    WHAT: Implements the architecture in Fig. 3 of Fu et al. 2025:
          Block1 (temporal conv) → Block2 (depthwise spatial conv) → Block3 (separable conv)
          → Squeeze → Multi-Head Sparse Attention → Squeeze & Distill → FC classifier.
    INPUT:
        x : Tensor (batch, 1, n_channels, n_timepoints) — raw epoched EEG
    OUTPUT:
        logits : Tensor (batch, n_classes) — class prediction scores
    """
    def __init__(self, n_channels=64, n_timepoints=641, n_classes=2,
                 n_heads=8, c=0.5, F1=16, D=1, F2=16, dropout=0.25):
        """
        Args:
            n_channels : int — number of EEG channels (64 for PhysioNet)
            n_timepoints : int — number of time samples per epoch
            n_classes : int — number of MI classes (2 for left/right)
            n_heads : int — number of attention heads (hyperparameter)
            c : float ∈ (0,1) — sparse attention coefficient (hyperparameter)
            F1 : int — number of temporal filters in Block1
            D : int — depth multiplier for depthwise convolution
            F2 : int — number of pointwise filters in Block3
            dropout : float — dropout probability
        """
        super().__init__()
        self.n_channels = n_channels
        self.n_timepoints = n_timepoints
        
        # ═══════════════════════════════════════════════════════════════
        # Block 1: Temporal Convolution
        # Paper: "8*C*T → 8*C*T" (Fig. 3) — learns frequency filters
        # Kernel spans half the sampling rate to capture ~2 Hz resolution
        # ═══════════════════════════════════════════════════════════════
        kernel_length = max(n_timepoints // 4, 16)  # temporal kernel size
        self.block1 = nn.Sequential(
            # Temporal convolution: (1, n_ch, T) → (F1, n_ch, T)
            nn.Conv2d(1, F1, (1, kernel_length),
                      padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),  # normalize activations for stable training
        )
        
        # ═══════════════════════════════════════════════════════════════
        # Block 2: Depthwise Spatial Convolution
        # Paper: "reduces the dimensionality of channel information from
        # 8 channels to 1" — learns spatial filters across electrodes
        # Uses groups=F1 so each filter operates independently
        # ═══════════════════════════════════════════════════════════════
        self.block2 = nn.Sequential(
            # Depthwise conv: (F1, n_ch, T) → (F1*D, 1, T)
            nn.Conv2d(F1, F1 * D, (n_channels, 1),
                      groups=F1, bias=False),  # groups=F1 makes it depthwise
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),                          # ELU for smooth gradients (better than ReLU for EEG)
            # Average pooling: compress time dimension by factor 4
            # Paper: "compresses the sample point data from 3072 to 768"
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),               # regularization
        )
        
        # ═══════════════════════════════════════════════════════════════
        # Block 3: Separable Convolution
        # Paper: "applies convolution only along the time dimension"
        # Depthwise temporal conv + pointwise 1x1 conv (= separable conv)
        # ═══════════════════════════════════════════════════════════════
        sep_kernel = max(kernel_length // 4, 4)  # shorter kernel for higher-level features
        self.block3 = nn.Sequential(
            # Depthwise temporal conv: (F1*D, 1, T/4) → (F1*D, 1, T/4)
            nn.Conv2d(F1 * D, F1 * D, (1, sep_kernel),
                      padding=(0, sep_kernel // 2), groups=F1 * D, bias=False),
            # Pointwise 1x1 conv: (F1*D, 1, T/4) → (F2, 1, T/4)
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            # Average pooling: compress time further by factor 4
            # Paper: compressed to 192 * 512 for attention processing
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )
        
        # ═══════════════════════════════════════════════════════════════
        # Compute feature dimensions after conv blocks (for attention input)
        # ═══════════════════════════════════════════════════════════════
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_timepoints)  # dummy input
            dummy = self.block1(dummy)
            dummy = self.block2(dummy)
            dummy = self.block3(dummy)
            # dummy shape: (1, F2, 1, T_reduced)
            self.feat_dim = dummy.shape[-1]    # reduced time dimension
            self.n_features = F2               # number of feature channels
        
        # ═══════════════════════════════════════════════════════════════
        # Multi-Head Sparse Self-Attention (Sec. II-D, Algorithm 1)
        # Paper: "feature matrix is divided into 8 heads"
        # Operates on (batch, seq_len=feat_dim, d_model=F2)
        # ═══════════════════════════════════════════════════════════════
        # Ensure d_model is divisible by n_heads via projection
        self.attn_d_model = max(n_heads, (F2 // n_heads) * n_heads)  # round up to nearest multiple
        self.pre_attn_proj = nn.Linear(F2, self.attn_d_model) if F2 != self.attn_d_model else nn.Identity()
        self.attention = SparseMultiHeadAttention(
            d_model=self.attn_d_model,
            n_heads=n_heads,
            c=c
        )
        self.post_attn_proj = nn.Linear(self.attn_d_model, F2) if F2 != self.attn_d_model else nn.Identity()
        self.attn_norm = nn.LayerNorm(F2)  # layer norm after attention (stabilizes training)
        
        # ═══════════════════════════════════════════════════════════════
        # Squeeze & Distill + Fully Connected Classifier
        # Paper: "192*512 → 96*16 → 1536 → class num" (Fig. 3)
        # ═══════════════════════════════════════════════════════════════
        fc_input_dim = F2 * self.feat_dim  # flatten attention output
        self.classifier = nn.Sequential(
            nn.Linear(fc_input_dim, 128),   # squeeze: reduce dimensionality
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),       # final classification layer
        )
    
    def forward(self, x):
        """
        Forward pass through the full DSCMI network.
        
        INPUT:  x — Tensor (batch, 1, n_channels, n_timepoints)
        OUTPUT: logits — Tensor (batch, n_classes)
        """
        # ── Convolution backbone ──
        x = self.block1(x)   # temporal convolution → (B, F1, C, T)
        x = self.block2(x)   # depthwise spatial → (B, F1*D, 1, T/4)
        x = self.block3(x)   # separable conv → (B, F2, 1, T/16)
        
        # ── Reshape for attention ──
        # Squeeze the spatial dim (which is 1 after depthwise conv)
        x = x.squeeze(2)              # (B, F2, T_reduced)
        x = x.permute(0, 2, 1)        # (B, T_reduced, F2) — sequence format for attention
        
        # ── Multi-Head Sparse Self-Attention ──
        x_proj = self.pre_attn_proj(x)         # project to attention dim if needed
        attn_out = self.attention(x_proj)       # sparse attention (Algorithm 1)
        attn_out = self.post_attn_proj(attn_out)  # project back
        x = self.attn_norm(x + attn_out)       # residual connection + layer norm
        
        # ── Classify ──
        x = x.reshape(x.size(0), -1)           # flatten: (B, F2 * T_reduced)
        logits = self.classifier(x)             # (B, n_classes)
        
        return logits


# ── Test model instantiation ──
n_ch, n_tp = X_demo.shape[1], X_demo.shape[2]
model_test = DSCMINet(n_channels=n_ch, n_timepoints=n_tp, n_classes=2).to(DEVICE)

# Count parameters
n_params = sum(p.numel() for p in model_test.parameters())
print(f"DSCMI model: {n_params:,} parameters")

# Forward pass test
x_test = torch.randn(2, 1, n_ch, n_tp).to(DEVICE)  # dummy batch
logits_test = model_test(x_test)
print(f"Input: {x_test.shape} → Output: {logits_test.shape}")
print("Section 6.3 complete — DSCMI architecture verified.")

## Section 6.4 — Training & Evaluation Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Train the model for one epoch.

    WHY:  Standard mini-batch SGD training loop with loss accumulation.
    INPUT:
        model : nn.Module — the DSCMI network
        loader : DataLoader — training data batches
        optimizer : torch.optim — parameter optimizer
        criterion : nn.Module — loss function (CrossEntropyLoss)
        device : torch.device — CPU or CUDA
    OUTPUT:
        avg_loss : float — mean training loss over all batches
        accuracy : float — training accuracy
    """
    model.train()  # set to training mode (enables dropout, batchnorm running stats)
    total_loss = 0.0
    correct = 0
    total = 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)  # move to GPU if available
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()          # clear accumulated gradients
        logits = model(X_batch)        # forward pass
        loss = criterion(logits, y_batch)  # compute cross-entropy loss
        loss.backward()                # backpropagate gradients
        optimizer.step()               # update parameters
        
        total_loss += loss.item() * X_batch.size(0)  # accumulate weighted loss
        preds = logits.argmax(dim=1)   # predicted class = argmax of logits
        correct += (preds == y_batch).sum().item()  # count correct predictions
        total += X_batch.size(0)
    
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """
    Evaluate model on a dataset (validation or test).

    WHY:  Computes loss and accuracy without gradient computation for efficiency.
    INPUT:
        model, loader, criterion, device — same as train_one_epoch
    OUTPUT:
        avg_loss : float — mean loss
        accuracy : float — classification accuracy
        all_preds : ndarray — predicted labels
        all_labels : ndarray — true labels
        all_probs : ndarray — softmax probabilities for ROC analysis
    """
    model.eval()  # set to evaluation mode (disables dropout)
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():  # no gradient computation during evaluation
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            probs = torch.softmax(logits, dim=1)  # convert logits to probabilities
            preds = logits.argmax(dim=1)
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    
    total = sum(len(p) for p in all_preds)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)
    accuracy = (all_preds == all_labels).mean()
    
    return total_loss / total, accuracy, all_preds, all_labels, all_probs


def train_model(model, train_loader, val_loader, device,
                n_epochs=300, lr=1e-3, patience=30):
    """
    Full training loop with early stopping and LR scheduling.

    WHY:  Early stopping prevents overfitting by monitoring validation loss.
          ReduceLROnPlateau adapts learning rate when progress stalls.
    INPUT:
        model : nn.Module — DSCMI network (will be modified in place)
        train_loader, val_loader : DataLoader — training and validation data
        device : torch.device
        n_epochs : int — maximum training epochs
        lr : float — initial learning rate
        patience : int — early stopping patience (epochs without improvement)
    OUTPUT:
        history : dict — training/validation loss and accuracy per epoch
        best_model_state : dict — state_dict of best model (lowest val loss)
    """
    criterion = nn.CrossEntropyLoss()  # standard classification loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)  # AdamW-style L2 reg
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, verbose=False
    )  # halve LR after 10 epochs without val loss improvement
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0
    
    for epoch in range(n_epochs):
        # Training step
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        # Validation step
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, device)
        
        scheduler.step(val_loss)  # adjust LR based on validation loss
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())  # save best weights
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break  # stop training — validation loss hasn't improved
    
    # Restore best model weights
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return history, best_model_state


print("Training utilities defined.")

## Section 6.5 — Automated Subject-Specific Hyperparameter Selection

**Addresses the paper's stated limitation** (Discussion, p.14):
> *"the optimal decoding parameters may vary between individuals. While subject-specific calibration is time-consuming, automated procedures for hyperparameter selection and adaptation could help reduce user burden."*

We use **Optuna with TPE (Tree-structured Parzen Estimator)** to jointly optimize:
- `n_heads` ∈ {2, 4, 8} — number of attention heads
- `c` ∈ (0.1, 0.9) — sparse attention coefficient
- `lr` ∈ [1e-4, 1e-2] — learning rate
- `dropout` ∈ [0.1, 0.5] — dropout rate

Evaluated via **repeated stratified 5-fold CV** (as in the paper) with mean validation accuracy as the objective.

In [ ]:
def make_loaders(X, y, train_idx, val_idx, batch_size=32):
    """
    Create PyTorch DataLoaders from numpy arrays and fold indices.

    WHY:  DataLoaders handle batching, shuffling, and GPU transfer efficiently.
    INPUT:
        X : ndarray (n_trials, n_ch, n_tp) — preprocessed EEG
        y : ndarray (n_trials,) — labels
        train_idx, val_idx : array — fold split indices
        batch_size : int
    OUTPUT:
        train_loader, val_loader : DataLoader
    """
    # Add channel dim for Conv2d: (trials, ch, tp) → (trials, 1, ch, tp)
    X_train = torch.FloatTensor(X[train_idx][:, np.newaxis, :, :])  # (N, 1, C, T)
    y_train = torch.LongTensor(y[train_idx])
    X_val = torch.FloatTensor(X[val_idx][:, np.newaxis, :, :])
    y_val = torch.LongTensor(y[val_idx])
    
    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True  # shuffle training data each epoch
    )
    val_loader = DataLoader(
        TensorDataset(X_val, y_val),
        batch_size=batch_size, shuffle=False  # no shuffling for evaluation
    )
    return train_loader, val_loader


def auto_hyperparam_search(X, y, n_trials=30, n_folds=5, n_repeats=2,
                           max_epochs=100, patience=15):
    """
    Automated Bayesian hyperparameter optimization using Optuna.

    WHY:  The paper notes that optimal hyperparameters vary per subject. Bayesian
          optimization (TPE) is more sample-efficient than grid search — it uses a
          probabilistic model to focus on promising regions of the search space.
    WHAT: For each trial, Optuna suggests hyperparameters. We evaluate them using
          repeated stratified k-fold CV (as in the paper) and report mean val accuracy.
    INPUT:
        X : ndarray (n_trials, n_ch, n_tp) — preprocessed EEG epochs
        y : ndarray (n_trials,) — labels
        n_trials : int — number of Optuna trials (search budget)
        n_folds : int — number of CV folds
        n_repeats : int — number of CV repetitions
        max_epochs : int — max training epochs per fold
        patience : int — early stopping patience
    OUTPUT:
        best_params : dict — optimal hyperparameters
        study : optuna.Study — full optimization history
    """
    n_ch, n_tp = X.shape[1], X.shape[2]
    
    def objective(trial):
        """
        Optuna objective: train DSCMI with suggested hyperparameters,
        return mean validation accuracy across CV folds.
        """
        # ── Suggest hyperparameters ──
        n_heads = trial.suggest_categorical('n_heads', [2, 4, 8])  # attention head count
        c = trial.suggest_float('c', 0.1, 0.9)            # sparse coefficient
        lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)  # learning rate (log scale)
        dropout = trial.suggest_float('dropout', 0.1, 0.5)  # dropout rate
        
        # ── Repeated Stratified K-Fold CV ──
        # Paper: "repeated stratified 5-fold cross-validation, in which the EEG trials
        # were randomly shuffled before each repetition"
        rskf = RepeatedStratifiedKFold(n_splits=n_folds, n_repeats=n_repeats,
                                        random_state=SEED)
        fold_accs = []  # collect validation accuracy from each fold
        
        for fold_idx, (train_idx, val_idx) in enumerate(rskf.split(X, y)):
            # Create data loaders for this fold
            train_loader, val_loader = make_loaders(X, y, train_idx, val_idx)
            
            # Instantiate fresh model with suggested hyperparameters
            model = DSCMINet(
                n_channels=n_ch, n_timepoints=n_tp, n_classes=len(np.unique(y)),
                n_heads=n_heads, c=c, dropout=dropout
            ).to(DEVICE)
            
            # Train with early stopping
            _, _ = train_model(model, train_loader, val_loader, DEVICE,
                               n_epochs=max_epochs, lr=lr, patience=patience)
            
            # Evaluate on validation fold
            criterion = nn.CrossEntropyLoss()
            _, val_acc, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)
            fold_accs.append(val_acc)
            
            # Prune unpromising trials early (Optuna's median pruning)
            trial.report(np.mean(fold_accs), fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        return np.mean(fold_accs)  # objective = mean CV accuracy
    
    # Create Optuna study — maximize validation accuracy
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),  # Bayesian TPE sampler
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=3)  # prune after 3 folds
    )
    
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    best_params = study.best_params
    print(f"Best hyperparameters: {best_params}")
    print(f"Best CV accuracy: {study.best_value:.4f}")
    
    return best_params, study


print("Automated hyperparameter search defined.")

## Section 6.6 — Run Experiments: Single Subject Demo + Multi-Subject

We first demonstrate the full pipeline on one subject, then run across multiple subjects.

In [ ]:
# ── Single-subject demonstration (Subject 1) ──
print("="*60)
print("Running automated hyperparameter search for Subject 1...")
print("="*60)

# Run Bayesian hyperparameter search
# Using fewer trials for demo speed — increase n_trials for production use
best_params_s1, study_s1 = auto_hyperparam_search(
    X_demo, y_demo,
    n_trials=15,      # Optuna trials (increase to 50+ for thorough search)
    n_folds=5,        # 5-fold CV as in paper
    n_repeats=1,      # 1 repeat for speed (paper uses repeated CV)
    max_epochs=100,   # max epochs per fold
    patience=15       # early stopping patience
)

print(f"\nOptimal params for Subject 1: {best_params_s1}")

In [ ]:
# ── Visualize hyperparameter search ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Optimization history — accuracy over trials
trials_df = study_s1.trials_dataframe()
completed = trials_df[trials_df['state'] == 'COMPLETE']
axes[0].plot(completed.index, completed['value'], 'bo-', markersize=4, label='Trial accuracy')
axes[0].axhline(study_s1.best_value, color='r', linestyle='--', label=f'Best: {study_s1.best_value:.3f}')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('CV Accuracy')
axes[0].set_title('Bayesian Optimization History')
axes[0].legend()

# Plot 2: Hyperparameter importance (if enough trials)
if len(completed) >= 5:
    try:
        importances = optuna.importance.get_param_importances(study_s1)
        params_names = list(importances.keys())
        params_values = list(importances.values())
        axes[1].barh(params_names, params_values, color='steelblue')
        axes[1].set_xlabel('Importance')
        axes[1].set_title('Hyperparameter Importance')
    except Exception:
        axes[1].text(0.5, 0.5, 'Insufficient data\nfor importance analysis',
                     ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('Hyperparameter Importance')

plt.tight_layout()
plt.savefig(FIG_DIR / 'section6_fig3_optuna.png', dpi=150)
plt.show()

### What to look for in this plot:
- **Left:** Optimization history should show accuracy improving (or stabilizing) over trials as TPE focuses on better regions
- **Right:** Hyperparameter importance — expect `n_heads` and `c` to be important (these are the DSCMI-specific params)

In [ ]:
# ── Final training with optimal hyperparameters ──
# Train on 80% of data, test on held-out 20%
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=SEED, stratify=y_demo
)  # stratified split preserves class balance

# Further split training into train/val for early stopping
X_trn, X_val, y_trn, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train
)

# Create data loaders
train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_trn[:, np.newaxis]), torch.LongTensor(y_trn)),
    batch_size=32, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_val[:, np.newaxis]), torch.LongTensor(y_val)),
    batch_size=32, shuffle=False
)
test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_test[:, np.newaxis]), torch.LongTensor(y_test)),
    batch_size=32, shuffle=False
)

# Instantiate model with optimal hyperparameters
n_ch, n_tp = X_demo.shape[1], X_demo.shape[2]
model_final = DSCMINet(
    n_channels=n_ch, n_timepoints=n_tp, n_classes=2,
    n_heads=best_params_s1['n_heads'],
    c=best_params_s1['c'],
    dropout=best_params_s1['dropout']
).to(DEVICE)

print(f"Training final DSCMI model with params: {best_params_s1}")
history, best_state = train_model(
    model_final, train_loader, val_loader, DEVICE,
    n_epochs=300, lr=best_params_s1['lr'], patience=30
)
print(f"Training finished after {len(history['train_loss'])} epochs.")

In [ ]:
# ── Plot training curves ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=1.5)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()

# Accuracy curves
axes[1].plot(history['train_acc'], label='Train Acc', linewidth=1.5)
axes[1].plot(history['val_acc'], label='Val Acc', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'section6_fig4_training.png', dpi=150)
plt.show()

### What to look for in this plot:
- **Loss:** Both curves should decrease; val loss flattening while train loss continues indicates early stopping was appropriate
- **Accuracy:** Val accuracy should stabilize near its maximum before early stopping triggers
- Large train-val gap → overfitting (may need more dropout or data augmentation)

In [ ]:
# ── Test set evaluation ──
criterion = nn.CrossEntropyLoss()
test_loss, test_acc, preds, labels, probs = evaluate(
    model_final, test_loader, criterion, DEVICE
)
kappa = cohen_kappa_score(labels, preds)  # Cohen's kappa — agreement beyond chance

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")
print("\nClassification Report:")
print(classification_report(labels, preds, target_names=['Left Fist', 'Right Fist']))

# ── Confusion matrix ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Left', 'Right'], yticklabels=['Left', 'Right'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title(f'Confusion Matrix (Acc={test_acc:.3f}, κ={kappa:.3f})')

# ROC curve
fpr, tpr, _ = roc_curve(labels, probs[:, 1])  # ROC for positive class (right fist)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)  # chance line
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'section6_fig5_results.png', dpi=150)
plt.show()

print(f"Section 6.6 complete — Subject 1 test accuracy: {test_acc:.4f}, Kappa: {kappa:.4f}")

### What to look for in this plot:
- **Confusion matrix:** Ideally diagonal-dominant; off-diagonal cells show misclassification patterns
- **ROC curve:** AUC > 0.5 indicates above-chance performance; closer to 1.0 is better
- Compare with EEGNet baseline (paper: ~82% on PhysioNet) and DSCMI (paper: ~87%)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Multi-subject evaluation (Subjects 1-10 for tractable runtime)
# ══════════════════════════════════════════════════════════════════════
SUBJECTS = list(range(1, 11))  # subjects 1-10 (increase for full 109-subject eval)

results = []  # collect per-subject results

for subj in SUBJECTS:
    print(f"\n{'='*50}")
    print(f"Processing Subject {subj}")
    print(f"{'='*50}")
    
    # ── Load and preprocess ──
    try:
        raw_s, events_s, eid_s = load_physionet_subject(subj)
        X_s, y_s, _ = preprocess_raw(raw_s, events_s, eid_s)
    except Exception as e:
        print(f"  Skipping subject {subj}: {e}")
        continue
    
    if len(np.unique(y_s)) < 2 or len(y_s) < 20:
        print(f"  Skipping subject {subj}: insufficient data ({len(y_s)} trials)")
        continue
    
    print(f"  Data: {X_s.shape}, Classes: {np.bincount(y_s)}")
    
    # ── Automated hyperparameter search (quick: 10 trials) ──
    best_params, _ = auto_hyperparam_search(
        X_s, y_s,
        n_trials=10,     # quick search (increase for production)
        n_folds=5,
        n_repeats=1,
        max_epochs=80,
        patience=15
    )
    
    # ── Final train/test split ──
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X_s, y_s, test_size=0.2, random_state=SEED, stratify=y_s
    )
    X_trn_s, X_val_s, y_trn_s, y_val_s = train_test_split(
        X_train_s, y_train_s, test_size=0.15, random_state=SEED, stratify=y_train_s
    )
    
    trn_ld = DataLoader(TensorDataset(
        torch.FloatTensor(X_trn_s[:, np.newaxis]), torch.LongTensor(y_trn_s)
    ), batch_size=32, shuffle=True)
    val_ld = DataLoader(TensorDataset(
        torch.FloatTensor(X_val_s[:, np.newaxis]), torch.LongTensor(y_val_s)
    ), batch_size=32, shuffle=False)
    tst_ld = DataLoader(TensorDataset(
        torch.FloatTensor(X_test_s[:, np.newaxis]), torch.LongTensor(y_test_s)
    ), batch_size=32, shuffle=False)
    
    n_ch_s, n_tp_s = X_s.shape[1], X_s.shape[2]
    model_s = DSCMINet(
        n_channels=n_ch_s, n_timepoints=n_tp_s, n_classes=2,
        n_heads=best_params['n_heads'], c=best_params['c'],
        dropout=best_params['dropout']
    ).to(DEVICE)
    
    _, _ = train_model(model_s, trn_ld, val_ld, DEVICE,
                       n_epochs=200, lr=best_params['lr'], patience=25)
    
    _, acc_s, preds_s, labels_s, _ = evaluate(
        model_s, tst_ld, nn.CrossEntropyLoss(), DEVICE
    )
    kappa_s = cohen_kappa_score(labels_s, preds_s)
    
    results.append({
        'Subject': subj, 'Accuracy': acc_s, 'Kappa': kappa_s,
        'n_heads': best_params['n_heads'], 'c': best_params['c'],
        'lr': best_params['lr'], 'dropout': best_params['dropout']
    })
    print(f"  → Accuracy: {acc_s:.4f}, Kappa: {kappa_s:.4f}")

# ── Summary table ──
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("Multi-Subject Results Summary")
print("="*60)
print(results_df.to_string(index=False))
print(f"\nMean Accuracy: {results_df['Accuracy'].mean():.4f} ± {results_df['Accuracy'].std():.4f}")
print(f"Mean Kappa:    {results_df['Kappa'].mean():.4f} ± {results_df['Kappa'].std():.4f}")

In [ ]:
# ── Multi-subject results visualization ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot: accuracy per subject
axes[0].bar(results_df['Subject'].astype(str), results_df['Accuracy'],
            color='steelblue', edgecolor='black')
axes[0].axhline(results_df['Accuracy'].mean(), color='red', linestyle='--',
                label=f"Mean: {results_df['Accuracy'].mean():.3f}")
axes[0].set_xlabel('Subject')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('DSCMI Test Accuracy per Subject')
axes[0].set_ylim(0, 1)
axes[0].legend()

# Scatter: optimal hyperparameters per subject
scatter = axes[1].scatter(
    results_df['n_heads'], results_df['c'],
    c=results_df['Accuracy'], s=100, cmap='RdYlGn',
    edgecolors='black', vmin=0.4, vmax=1.0
)
plt.colorbar(scatter, ax=axes[1], label='Accuracy')
axes[1].set_xlabel('Number of Heads')
axes[1].set_ylabel('Sparse Coefficient c')
axes[1].set_title('Optimal Hyperparameters per Subject')

plt.tight_layout()
plt.savefig(FIG_DIR / 'section6_fig6_multisubject.png', dpi=150)
plt.show()

### What to look for in this plot:
- **Left:** Inter-subject variability in accuracy — confirms the paper's observation that decoding varies across individuals
- **Right:** Different subjects prefer different hyperparameters — justifies the automated subject-specific tuning approach
- Green dots (high accuracy) scattered across the parameter space → no single config is universally optimal

## Section 6.7 — Real-Time EEG Simulation Module

This module replays a saved EEG file **sample-by-sample** as if streamed from a real amplifier, enabling the trained DSCMI decoder to run in a closed-loop BCI simulation without physical hardware.

**Design principle: Hardware-agnostic decoder interface.**

The architecture uses an **abstract data source** pattern:
- `EEGSource` (abstract) → defines the interface: `get_sample()`, `get_chunk()`, `info`
- `FileReplaySource` → implements file-based streaming (PhysioNet EDF files)
- `LSLSource` (stub) → would implement LSL-based real hardware streaming

The **decoder pipeline** (`RealtimeBCISession`) only depends on the `EEGSource` interface — swapping file replay for real hardware (e.g., pylsl or brainflow) requires **zero changes** to the decoder or preprocessing logic.

In [ ]:
from abc import ABC, abstractmethod


class EEGSource(ABC):
    """
    Abstract base class for EEG data sources.

    WHY:  Decouples the decoder from the data acquisition method. The same
          preprocessing + decoding pipeline works with file replay, LSL streams,
          BrainFlow, or any other source that implements this interface.
    WHAT: Defines the contract that all EEG sources must fulfill.
    """
    @property
    @abstractmethod
    def sfreq(self) -> float:
        """Sampling frequency in Hz."""
        pass
    
    @property
    @abstractmethod
    def ch_names(self) -> list:
        """List of channel names."""
        pass
    
    @property
    @abstractmethod
    def n_channels(self) -> int:
        """Number of channels."""
        pass
    
    @abstractmethod
    def get_chunk(self, n_samples: int) -> np.ndarray:
        """
        Get a chunk of n_samples from the source.
        
        INPUT:  n_samples : int — number of time samples to read
        OUTPUT: ndarray (n_channels, n_samples) — EEG data chunk
        """
        pass
    
    @abstractmethod
    def is_active(self) -> bool:
        """Whether the source still has data to provide."""
        pass


class FileReplaySource(EEGSource):
    """
    Replays a saved EEG file sample-by-sample as if streaming live.

    WHY:  Enables testing the full BCI pipeline on stored data (e.g., from stroke
          patients) without needing physical EEG hardware. Simulates real-time
          by advancing a read pointer through the data.
    WHAT: Loads an MNE-compatible EEG file and serves samples sequentially.
    INPUT:
        raw : mne.io.Raw — loaded and optionally preprocessed EEG data
        real_time : bool — if True, add delays to simulate real-time speed
    """
    def __init__(self, raw, real_time=False):
        self._raw = raw  # store reference to MNE Raw object
        self._data = raw.get_data()  # (n_channels, n_total_samples) — full data matrix
        self._pos = 0  # current read position (sample index)
        self._real_time = real_time  # whether to sleep between chunks
    
    @property
    def sfreq(self) -> float:
        return self._raw.info['sfreq']
    
    @property
    def ch_names(self) -> list:
        return self._raw.ch_names
    
    @property
    def n_channels(self) -> int:
        return len(self._raw.ch_names)
    
    def get_chunk(self, n_samples: int) -> np.ndarray:
        """
        Read next n_samples from the file.
        
        INPUT:  n_samples : int — samples to read
        OUTPUT: ndarray (n_channels, n_samples) — EEG chunk
        """
        end = min(self._pos + n_samples, self._data.shape[1])
        chunk = self._data[:, self._pos:end]  # slice from current position
        
        if self._real_time:
            # Sleep to simulate real-time data arrival
            time.sleep(n_samples / self.sfreq)
        
        self._pos = end  # advance read pointer
        return chunk
    
    def is_active(self) -> bool:
        return self._pos < self._data.shape[1]
    
    def reset(self):
        """Reset read pointer to beginning (for repeated replay)."""
        self._pos = 0


class LSLSource(EEGSource):
    """
    Placeholder for real LSL hardware streaming.
    
    WHY:  Demonstrates that the decoder interface is hardware-agnostic. To connect
          real hardware, implement get_chunk() using pylsl.StreamInlet.pull_chunk().
    NOTE: Not executed — requires pylsl and physical hardware.
    """
    def __init__(self, stream_name='EEG', n_channels=64, sfreq=160.0):
        self._sfreq = sfreq
        self._n_channels = n_channels
        self._ch_names = [f'Ch{i}' for i in range(n_channels)]
        # In real implementation:
        # from pylsl import StreamInlet, resolve_stream
        # streams = resolve_stream('name', stream_name)
        # self._inlet = StreamInlet(streams[0])
    
    @property
    def sfreq(self): return self._sfreq
    @property
    def ch_names(self): return self._ch_names
    @property
    def n_channels(self): return self._n_channels
    
    def get_chunk(self, n_samples):
        # Real implementation: samples, _ = self._inlet.pull_chunk(max_samples=n_samples)
        raise NotImplementedError("Connect real LSL stream to use this source.")
    
    def is_active(self):
        return True  # hardware streams are always active until disconnected


print("EEG source interfaces defined (FileReplaySource, LSLSource stub).")

In [ ]:
class RingBuffer:
    """
    Circular buffer for accumulating streaming EEG samples.

    WHY:  In real-time BCI, we need to maintain a sliding window of recent EEG data.
          A ring buffer provides O(1) append and avoids memory reallocation.
    INPUT:
        n_channels : int — number of EEG channels
        max_samples : int — buffer capacity (window size in samples)
    """
    def __init__(self, n_channels, max_samples):
        self.buffer = np.zeros((n_channels, max_samples))  # pre-allocated storage
        self.max_samples = max_samples
        self.write_pos = 0  # next write position
        self.filled = False  # whether buffer has been completely filled once
    
    def append(self, chunk):
        """
        Append a chunk of data to the ring buffer.
        
        INPUT: chunk — ndarray (n_channels, n_new_samples)
        """
        n_new = chunk.shape[1]
        
        if n_new >= self.max_samples:
            # Chunk larger than buffer: just keep the last max_samples
            self.buffer[:] = chunk[:, -self.max_samples:]
            self.write_pos = 0
            self.filled = True
        else:
            end_pos = self.write_pos + n_new
            if end_pos <= self.max_samples:
                # Fits without wrapping
                self.buffer[:, self.write_pos:end_pos] = chunk
            else:
                # Wraps around
                first = self.max_samples - self.write_pos  # samples before wrap
                self.buffer[:, self.write_pos:] = chunk[:, :first]
                self.buffer[:, :n_new - first] = chunk[:, first:]
                self.filled = True
            self.write_pos = end_pos % self.max_samples
            if end_pos >= self.max_samples:
                self.filled = True
    
    def get_data(self):
        """
        Get the current buffer contents in chronological order.
        
        OUTPUT: ndarray (n_channels, max_samples)
        """
        if not self.filled:
            return self.buffer[:, :self.write_pos]  # only filled portion
        # Reorder: oldest samples first
        return np.concatenate([
            self.buffer[:, self.write_pos:],
            self.buffer[:, :self.write_pos]
        ], axis=1)
    
    def is_ready(self):
        """Whether the buffer has accumulated enough data for one full window."""
        return self.filled


class RealtimeBCISession:
    """
    Real-time BCI session controller — hardware-agnostic.

    WHY:  Orchestrates the closed-loop BCI: stream EEG → buffer → preprocess → decode.
          The decoder and preprocessing are completely decoupled from the data source.
    WHAT: Reads chunks from any EEGSource, maintains a sliding window via RingBuffer,
          applies bandpass + z-score preprocessing, and runs DSCMI inference.
    INPUT:
        source : EEGSource — any data source implementing the EEGSource interface
        model : nn.Module — trained DSCMI model
        window_sec : float — classification window length in seconds
        stride_sec : float — stride between consecutive classifications (seconds)
        l_freq, h_freq : float — bandpass filter cutoffs (Hz)
    """
    def __init__(self, source, model, device,
                 window_sec=4.0, stride_sec=0.5,
                 l_freq=7.0, h_freq=35.0):
        self.source = source
        self.model = model
        self.device = device
        
        self.sfreq = source.sfreq                              # sampling rate
        self.window_samples = int(window_sec * self.sfreq)     # samples per window
        self.stride_samples = int(stride_sec * self.sfreq)     # samples per stride
        self.n_channels = source.n_channels
        
        # Ring buffer to accumulate streaming samples
        self.buffer = RingBuffer(self.n_channels, self.window_samples)
        
        # Design bandpass filter coefficients (applied in real-time)
        # Using SOS (second-order sections) for numerical stability
        nyq = self.sfreq / 2.0  # Nyquist frequency
        self.sos = sig.butter(4, [l_freq / nyq, h_freq / nyq],
                              btype='band', output='sos')  # 4th-order Butterworth
        
        # Store results
        self.predictions = []  # (timestamp, predicted_class, confidence)
        self.class_names = ['Left Fist', 'Right Fist']
    
    def _preprocess_window(self, window):
        """
        Apply real-time preprocessing to a single window.
        
        WHY:  Same preprocessing as offline pipeline but applied to a single window.
        INPUT:  window — ndarray (n_channels, n_timepoints)
        OUTPUT: x — Tensor (1, 1, n_channels, n_timepoints) ready for model
        """
        # Bandpass filter (causal, applied per-channel)
        filtered = sig.sosfilt(self.sos, window, axis=1)  # filter along time axis
        
        # Z-score normalization per channel
        for ch in range(filtered.shape[0]):
            mu = filtered[ch].mean()
            sigma = filtered[ch].std() + 1e-8
            filtered[ch] = (filtered[ch] - mu) / sigma
        
        # Convert to tensor with batch and conv-channel dims
        x = torch.FloatTensor(filtered).unsqueeze(0).unsqueeze(0)  # (1, 1, C, T)
        return x.to(self.device)
    
    def _decode(self, x):
        """
        Run DSCMI inference on a preprocessed window.
        
        INPUT:  x — Tensor (1, 1, n_channels, n_timepoints)
        OUTPUT: pred_class : int, confidence : float
        """
        self.model.eval()
        with torch.no_grad():
            logits = self.model(x)                    # (1, n_classes)
            probs = torch.softmax(logits, dim=1)      # convert to probabilities
            pred_class = probs.argmax(dim=1).item()   # predicted class index
            confidence = probs[0, pred_class].item()  # confidence of prediction
        return pred_class, confidence
    
    def run(self, max_predictions=50, verbose=True):
        """
        Run the real-time BCI session.

        WHY:  Main loop that simulates closed-loop BCI operation. Reads from source,
              fills buffer, classifies when enough data is accumulated.
        INPUT:
            max_predictions : int — stop after this many classifications
            verbose : bool — print predictions to console
        OUTPUT:
            predictions : list of (time_sec, pred_class, confidence) tuples
        """
        self.predictions = []
        sample_count = 0  # total samples processed
        last_classify = -self.stride_samples  # force first classification after buffer fills
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Starting Real-Time BCI Session")
            print(f"  Source: {type(self.source).__name__}")
            print(f"  Window: {self.window_samples/self.sfreq:.1f}s, Stride: {self.stride_samples/self.sfreq:.1f}s")
            print(f"  Channels: {self.n_channels}, Sfreq: {self.sfreq} Hz")
            print(f"{'='*60}\n")
        
        while self.source.is_active() and len(self.predictions) < max_predictions:
            # Read a chunk (stride-sized) from the source
            chunk = self.source.get_chunk(self.stride_samples)
            
            if chunk.shape[1] == 0:
                break  # source exhausted
            
            # Append to ring buffer
            self.buffer.append(chunk)
            sample_count += chunk.shape[1]
            
            # Classify when buffer is full and stride interval has elapsed
            if self.buffer.is_ready() and (sample_count - last_classify) >= self.stride_samples:
                # Get current window from buffer
                window = self.buffer.get_data()  # (n_channels, window_samples)
                
                # Preprocess and classify
                x = self._preprocess_window(window)
                pred_class, confidence = self._decode(x)
                
                # Record prediction
                time_sec = sample_count / self.sfreq
                self.predictions.append((time_sec, pred_class, confidence))
                last_classify = sample_count
                
                if verbose:
                    print(f"  t={time_sec:7.2f}s | Prediction: {self.class_names[pred_class]:12s} "
                          f"| Confidence: {confidence:.3f}")
        
        if verbose:
            print(f"\nSession complete: {len(self.predictions)} predictions made.")
        
        return self.predictions


print("RealtimeBCISession defined — hardware-agnostic closed-loop BCI.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Run the real-time simulation using Subject 1's data and trained model
# ══════════════════════════════════════════════════════════════════════

# Create a file replay source from the same raw data
source = FileReplaySource(raw_demo, real_time=False)  # fast replay (no delays)

# The trained model expects (1, n_channels, n_timepoints) — but the stream
# window size must match. Use the same window as our epochs.
print(f"Model expects: {n_ch} channels × {n_tp} timepoints")
print(f"Source provides: {source.n_channels} channels × {source.sfreq} Hz")

# Create and run the BCI session
session = RealtimeBCISession(
    source=source,
    model=model_final,
    device=DEVICE,
    window_sec=n_tp / source.sfreq,   # match epoch length
    stride_sec=0.5,                    # classify every 0.5 seconds
    l_freq=7.0, h_freq=35.0           # same filter as preprocessing
)

# Run simulation — 30 predictions for demo
predictions = session.run(max_predictions=30, verbose=True)

In [ ]:
# ── Visualize real-time predictions ──
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

if predictions:
    times = [p[0] for p in predictions]       # timestamps
    classes = [p[1] for p in predictions]      # predicted classes
    confs = [p[2] for p in predictions]        # confidence scores
    
    # Plot 1: Predicted class over time
    colors = ['#2196F3' if c == 0 else '#FF5722' for c in classes]  # blue=left, red=right
    axes[0].scatter(times, classes, c=colors, s=60, edgecolors='black', zorder=3)
    axes[0].set_yticks([0, 1])
    axes[0].set_yticklabels(['Left Fist', 'Right Fist'])
    axes[0].set_title('Real-Time BCI Predictions (File Replay Simulation)')
    axes[0].set_ylabel('Predicted Class')
    axes[0].axhline(0.5, color='gray', linestyle=':', alpha=0.5)
    
    # Plot 2: Confidence over time
    axes[1].bar(times, confs, width=0.4, color=colors, edgecolor='black', alpha=0.8)
    axes[1].axhline(0.5, color='red', linestyle='--', label='Chance level', alpha=0.7)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Confidence')
    axes[1].set_title('Prediction Confidence')
    axes[1].set_ylim(0, 1)
    axes[1].legend()
else:
    axes[0].text(0.5, 0.5, 'No predictions generated', ha='center', va='center',
                 transform=axes[0].transAxes, fontsize=14)

plt.tight_layout()
plt.savefig(FIG_DIR / 'section6_fig7_realtime.png', dpi=150)
plt.show()

### What to look for in this plot:
- **Top:** Predicted motor imagery class at each time step — should show temporal patterns (not random noise)
- **Bottom:** Confidence bars — higher bars indicate more certain predictions
- During MI events (T1/T2), predictions should cluster consistently; during rest, predictions may be less confident
- Blue = left fist, Orange/Red = right fist

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Demonstrate hardware-agnostic design: show that swapping source
# requires NO changes to decoder or preprocessing
# ══════════════════════════════════════════════════════════════════════

print("Demonstrating hardware-agnostic design:")
print("="*50)
print()
print("To swap from file replay to real hardware, simply change:")
print()
print("  # File replay (current):")
print("  source = FileReplaySource(raw, real_time=False)")
print()
print("  # Real LSL hardware (swap):")
print("  # source = LSLSource(stream_name='MyEEG', n_channels=64, sfreq=160)")
print()
print("  # BrainFlow (swap):")
print("  # source = BrainFlowSource(board_id=0)  # implement EEGSource interface")
print()
print("The decoder and preprocessing remain UNCHANGED:")
print("  session = RealtimeBCISession(source=source, model=model, ...)")
print("  predictions = session.run()")
print()

# Verify: RealtimeBCISession accepts any EEGSource subclass
assert isinstance(source, EEGSource), "FileReplaySource must implement EEGSource"
print("✓ FileReplaySource is a valid EEGSource — interface verified.")
print(f"\nSection 6.7 complete — real-time simulation with {len(predictions)} predictions.")

## Section 6.8 — Summary

This notebook implemented:

1. **DSCMI Decoder** — depthwise separable convolution + multi-head sparse self-attention (Algorithm 1, Eqs. 1-10)
2. **Full preprocessing pipeline** — 7-35 Hz FIR bandpass → ICA artifact removal → epoching → z-score normalization
3. **Automated subject-specific hyperparameter selection** — Bayesian optimization (Optuna TPE) over {n_heads, c, lr, dropout}
4. **Real-time EEG simulation** — hardware-agnostic `EEGSource` interface + `RealtimeBCISession` closed-loop controller

In [ ]:
# ── Final summary ──
print("="*60)
print("SECTION 6 COMPLETE — DSCMI Decoder Implementation")
print("="*60)
print(f"\nSubject 1 Results:")
print(f"  Test Accuracy:  {test_acc:.4f}")
print(f"  Cohen's Kappa:  {kappa:.4f}")
print(f"  Best Params:    {best_params_s1}")
print(f"\nMulti-Subject Results (n={len(results_df)}):")
print(f"  Mean Accuracy:  {results_df['Accuracy'].mean():.4f} ± {results_df['Accuracy'].std():.4f}")
print(f"  Mean Kappa:     {results_df['Kappa'].mean():.4f} ± {results_df['Kappa'].std():.4f}")
print(f"\nReal-Time Simulation:")
print(f"  Predictions:    {len(predictions)}")
if predictions:
    avg_conf = np.mean([p[2] for p in predictions])
    print(f"  Avg Confidence: {avg_conf:.4f}")
print(f"\nModel Parameters: {n_params:,}")
print(f"Device: {DEVICE}")
print("\nSection 6 complete — all components verified.")